## Mars Photogrammetry Preprocessing Pipeline

By Christian Tate, Cornell University; Ithaca, NY

https://github.com/cdt59/MPPP


To install the python stand-alone metashape package, visit this [site](v) and download the .whl for your system, then run the command `!pip install Metashape-<version number>-cp37.cp38.cp39.cp310.cp311-none-win_amd64.whl`

In [1]:
# Import python modules

import numpy as np
import cv2, glob, time, os
import matplotlib.pyplot as plt
from numpy.linalg import inv
from scipy import stats
import time
# from planetaryimage import PDS3Image
# import colour_demosaicing
# from PIL import Image
# import matplotlib.cm as cm
# from scipy import interpolate
# from scipy.spatial.transform import Rotation as R
# import colour_demosaicing
import pandas as pd
from scipy.optimize import curve_fit

import Metashape
np.set_printoptions(suppress=True)

from src2.cmod import *
from src2.Image import Image


# %run src/MPPP.py

%matplotlib inline

directory_input  = 'Z:/Mastcam-Z/agisoft/data/'
# directory_input  = 'C:/Users/cdt59/Desktop/agisoft/data'
# directory_input  = 'C:/Users/cdt59/Desktop/agisoft/data/cal'


In [2]:
import os
os.path.isfile("Z:/Mastcam-Z/agisoft/jezero/m41_site_cal_v231.psx")

True

In [4]:
doc = Metashape.Document( read_only=1 )

# doc.open(path="C:/Users/cdt59/Desktop/agisoft/m32_site_cal690_v2.psx")
# doc.open(path="Z:/Mastcam-Z/agisoft/jezero/m32_site_cal690_.psx")
# doc.open(path="Z:/Mastcam-Z/agisoft/jezero/m32_site_cal690_.psx")
# doc.open(path="C:/Users/cdt59/Desktop/agisoft/m41_site_cal_v202.psx")
# doc.open(path="C:/Users/cdt59/Desktop/agisoft/m41_site_cal_v4.psx")
# doc.open(path="C:/Users/cdt59/Desktop/agisoft/m41_site_cal_v217_.psx")
# doc.open(path="Z:/Mastcam-Z/agisoft/jezero/m41_site_cal_v217_v2_.psx")
# doc.open(path="Z:/Mastcam-Z/agisoft/jezero/m41_site_cal_v222_.psx")
# doc.open(path="C:/Users/cdt59/Desktop/agisoft/m41_site_cal_v224_.psx")
doc.open(path="Z:/Mastcam-Z/agisoft/jezero/m41_site_cal_v231.psx")

# path_references = "C:/Users/cdt59/Desktop/agisoft/m32_site_cal690_rnav_v2.txt"
# path_references = "C:/Users/cdt59/Desktop/agisoft/positions_refs_00_zcam_rnav.txt"
# path_references = "C:/Users/cdt59/Desktop/agisoft/positions_refs_00_zcam_rnav_v4.txt"
# path_references = "C:/Users/cdt59/Desktop/agisoft/positions_refs_00_zcam_rnav_v217.txt"
# path_references = "Z:/Mastcam-Z/agisoft/jezero/positions_refs_zcam_rnav_v217_v2.txt"
# path_references = "C:/Users/cdt59/Desktop/agisoft/m41_site_cal_v224_refs_v4.txt"

# df_r = pd.read_csv( path_references, skiprows=1 )
# df_r

version_name = f'm41_site_cal_v230_v4_{time.strftime("%Y%m%d_%H%M%S")}'

save_path =  '/plots/'+ version_name + '.xlsx'

# doc.chunk.__dir__()

ref_df = pd.read_csv("m41_site_cal_v231_refs_v1.txt", sep="\t")
ref_df  = ref_df[["#Label", "Omega", "Phi", "Kappa", "X", "Y", "Z",
                "Omega_est", "Phi_est", "Kappa_est", "X_est", "Y_est", "Z_est"]]

In this notebook, the estimated OPK values that represent the transformation from the camera frame to the rover frame exported from agisoft metashape is used to transform the $H^c_{est}$ matrix, which was derived using $fl_{est}$, $b_{1,est}$, $b_{2,est}$, $c_{x,est}$ and $c_{y,est}$ once again from metashape, to $H^r_{est}$. $C^r_{est}$ was the $X_{est}$, $Y_{est}$ and $Z_{est}$ estimated by metashape. The $o^r_{est}$ was assumed to be equal to $a^r_{est}$, lastly $r^r_{est}$ is the distortion $[0, k_1, k_2]$ and is assumed to not be transformed.



In [5]:
%%time

# %run src/MPPP.py
# idxs = [41888, 41887, 41886, 41885, 41884, 41883, 41882, 41881, 41880,
#     41879, 41878, 41877, 41876, 41875, 41874, 41873, 41872, 41871, 41870,
#     41869, 41868, 41867, 41866, 41865, 41864, 41863, 41862, 41861, 41860,
#     41859, 41858, 41857, 41856, 41855, 41854, 41853, 41852, 41851, 41850, 
#     41849, 41848, 41847, 41846, 41845, 41844, 41843, 41842, 41841, 41840, 
#     41839, 41838, 41837, 41836, 41835, 41834, 41833, 41832, 41831, 41830, 41829]

N_cams = len( doc.chunk.cameras )

columns = [ 'name', 'zcam', 'cam', 'fl', 
            'temp_DEA', 'temp_FPA', 
            'temp_HTR1', 'temp_HTR2', 
            'fmc', 'zmc', 'filtmc', 
            'az_RSM', 'el_RSM', 
            'c_r','a_r','h_r','v_r','o_r','r_r',
            'c_r_est','a_r_est','h_r_est','v_r_est','o_r_est','r_r_est',
            'f_est','cx_est','cy_est','b1_est','b2_est',
            'k0_est','k1_est','k2_est','k3_est','k4_est','p1_est','p2_est',
            'f','cx','cy','b1','b2',
            'k0', 'k1','k2','k3','k4','p1','p2',
            'w','h',
            'X_ref','Y_ref','Z_ref','X_est','Y_est','Z_est',
            'KnKR_e_c_est',
            'q_rpm', 'R_rpm', 't_rpm']
ls = []
for i in range( N_cams )[::-1]:
    if doc.chunk.cameras[ i ].enabled and doc.chunk.cameras[ i ].label[:1] == 'Z':
        cam = doc.chunk.cameras[ i ]
        IMG_path = glob.glob( directory_input + '/zcam/*/' + cam.label[:54] + '.IMG' ) + glob.glob( directory_input + '/zcam/' + cam.label[:54] + '.IMG' )
        
        if len(IMG_path) and cam.transform and cam.reference.enabled:
            print( i, cam)
            
            cam = doc.chunk.cameras[i]
            cam_label = cam.label

            zcam = 0 if cam_label[1] == "L" else 1
            cam_name = cam_label[:2] + cam_label[45:48]
            fl = int(cam_label[45:48])

            #======================   Metashape   =====================
            # Rotation from camera to rover using agisoft estimated OPK values
            opk = ref_df.iloc[i, 7:10] + np.array([0,0,-90])
            R_rc_agi = R.from_euler('XYZ', opk, degrees=1).as_matrix()
            R_rc_agi  = np.array([[0, 1, 0], [1, 0, 0], [0, 0, -1]]) @ R_rc_agi @ np.array([[0, 1, 0], [1, 0, 0], [0, 0, -1]])
            t_rc_agi = np.array([[0, 1, 0], [1, 0, 0], [0, 0, -1]]) @ ref_df.iloc[i, 10:13].to_list()

            # h, v & a vectors in agisoft camera frame
            fl = cam.calibration.f
            cx, cy = cam.calibration.cx+ (1648//2), cam.calibration.cy + (1200//2)
            b1, b2 = cam.calibration.b1, cam.calibration.b2
            hva_cam = np.array([[fl+b1, 0, 0], [b2, fl, 0], [cx, cy, 1]])

            # transform from agisoft camera frame to rover frame
            c_r_est = t_rc_agi
            hva_r_est = R_rc_agi @ hva_cam
            h_r_est, v_r_est, a_r_est  = hva_r_est[:,0], hva_r_est[:,1], hva_r_est[:,2]
            o_r_est = a_r_est.copy()
            k1, k2 = cam.calibration.k1, cam.calibration.k2
            r_r_est = np.array([0, k1, k2])


            #======================   PDS Label   =====================
            # parse cahvor from PDS label
            IMG_path = glob.glob( directory_input + '/zcam/*/' + cam.label[:54] + '.IMG' ) +\
                        glob.glob( directory_input + '/zcam/' + cam.label[:54] + '.IMG' )
            im = Image(IMG_path=IMG_path[0])
            c_r = np.array(im.label['GEOMETRIC_CAMERA_MODEL']['MODEL_COMPONENT_1'])
            a_r = np.array(im.label['GEOMETRIC_CAMERA_MODEL']['MODEL_COMPONENT_2'])
            h_r = np.array(im.label['GEOMETRIC_CAMERA_MODEL']['MODEL_COMPONENT_3'])
            v_r = np.array(im.label['GEOMETRIC_CAMERA_MODEL']['MODEL_COMPONENT_4'])
            o_r = np.array(im.label['GEOMETRIC_CAMERA_MODEL']['MODEL_COMPONENT_5'])
            r_r = np.array(im.label['GEOMETRIC_CAMERA_MODEL']['MODEL_COMPONENT_6'])

            # Extrinsic and intrinsic camera parameters from PDS labels
            x, y, z = c_r[0], c_r[1], c_r[2]

            hs = np.linalg.norm(np.cross(a_r, h_r), ord=2)
            vs = np.linalg.norm(np.cross(a_r, v_r), ord=2)
            hc = np.dot(h_r, a_r)
            vc = np.dot(v_r, a_r)

            hp = (h_r-hc*a_r)/hs
            vp = (v_r-vc*a_r)/vs
            norm_hp = np.linalg.norm(hp, ord=2)
            norm_vp = np.linalg.norm(vp, ord=2)

            sin_theta = np.clip(np.linalg.norm( np.cross(vp, hp)), a_min=-1, a_max=1)
            cos_theta = np.sqrt(1-sin_theta**2)

            fl = vs
            b1 = - (hs * (-sin_theta)) - vs
            b2 = hs * cos_theta
            cx = hc
            cy = vc
            k0, k1, k2 = r_r[0], r_r[1], r_r[2]


            # random things for logging
            KnKR_est_ = R_rc_agi
            t_est = t_rc_agi
            t_ref = np.array(cam.reference.location)
            q_RM = q_wxyz2xyzw( im.label['GEOMETRIC_CAMERA_MODEL']['MODEL_TRANSFORM_QUATERNION'] )
            R_RM = R.from_quat( q_RM ).as_matrix()
            t_RM = im.label['GEOMETRIC_CAMERA_MODEL']['MODEL_TRANSFORM_VECTOR']

            l = [cam.label, zcam, cam_name, int(cam_label[45:48]),
                im.label['INSTRUMENT_STATE_PARMS']['INSTRUMENT_TEMPERATURE'][0], im.label['INSTRUMENT_STATE_PARMS']['INSTRUMENT_TEMPERATURE'][1], 
                im.label['INSTRUMENT_STATE_PARMS']['INSTRUMENT_TEMPERATURE'][2], im.label['INSTRUMENT_STATE_PARMS']['INSTRUMENT_TEMPERATURE'][3], 
                im.focus_mc, im.zoom_mc, im.filter_mc, 
                im.label['RSM_ARTICULATION_STATE']['ARTICULATION_DEVICE_ANGLE'][0], im.label['RSM_ARTICULATION_STATE']['ARTICULATION_DEVICE_ANGLE'][1],
                list(c_r), list(a_r), list(h_r), list(v_r), list(o_r), list(r_r),
                list(c_r_est), list(a_r_est), list(h_r_est), list(v_r_est), list(o_r_est), list(r_r_est),
                cam.calibration.f, cam.calibration.cx+(1648//2), cam.calibration.cy+(1200//2), cam.calibration.b1, cam.calibration.b2,\
                0, cam.calibration.k1, cam.calibration.k2, cam.calibration.k3, cam.calibration.k4, cam.calibration.p1, cam.calibration.p2, 
                fl, cx, cy, b1, b2, 
                k0, k1, k2, 0, 0, 0, 0, 
                cam.calibration.width,cam.calibration.height,
                t_ref[0], t_ref[1], t_ref[2], t_est[0], t_est[1], t_est[2],
                KnKR_est_.tolist(), list(q_RM), R_RM.tolist(), list(t_RM)
                ]
            ls.append( l )

df = pd.DataFrame(ls, columns = columns )

df.to_excel('plots/'+ version_name + '.xlsx')

43757 <Camera 'ZR0_1178_0771519262_818RAD_N0540000ZCAM05192_0630LMA01'>
43756 <Camera 'ZR0_1178_0771519247_818RAD_N0540000ZCAM05192_0630LMA01'>
43755 <Camera 'ZR0_1178_0771519231_818RAD_N0540000ZCAM05192_0630LMA01'>
43754 <Camera 'ZR0_1178_0771519212_818RAD_N0540000ZCAM05192_0630LMA01'>
43753 <Camera 'ZR0_1178_0771519195_818RAD_N0540000ZCAM05192_0630LMA01'>
43752 <Camera 'ZR0_1178_0771519178_818RAD_N0540000ZCAM05192_0630LMA01'>
43751 <Camera 'ZR0_1178_0771519164_818RAD_N0540000ZCAM05192_0630LMA01'>
43750 <Camera 'ZR0_1178_0771519146_818RAD_N0540000ZCAM05192_0630LMA01'>
43749 <Camera 'ZR0_1178_0771519128_818RAD_N0540000ZCAM05192_0630LMA01'>
43748 <Camera 'ZR0_1178_0771519112_818RAD_N0540000ZCAM05192_0630LMA01'>
43747 <Camera 'ZR0_1178_0771519064_303RAD_N0540000ZCAM09220_0630LMA01'>
43746 <Camera 'ZR0_1178_0771519051_269RAD_N0540000ZCAM09220_0630LMA01'>
43745 <Camera 'ZR0_1178_0771519036_269RAD_N0540000ZCAM09220_0630LMA01'>
43744 <Camera 'ZR0_1178_0771519021_277RAD_N0540000ZCAM09220_0630

In [6]:
df = pd.DataFrame(ls, columns = columns )

df.to_excel('plots/'+ version_name + '.xlsx')

In [9]:
'plots/'+ version_name + '.xlsx'

'plots/m41_site_cal_v230_v4_20240815_125829.xlsx'

In [8]:
i = 43743

cam = doc.chunk.cameras[i]
cam_label = cam.label
print(cam_label)

zcam = 0 if cam_label[1] == "L" else 1
cam_name = cam_label[:2] + cam_label[45:48]
fl = int(cam_label[45:48])

#======================   Agisoft Metashape   =====================
opk = ref_df.iloc[i, 7:10].tolist()
opk_p = opk + np.array([0,0,-90])
R_p = R.from_euler('XYZ', opk_p, degrees=1).as_matrix()
print(f"R_agi pre-transform:\n{R_p}")
R_rc_agi  = np.array([[0, 1, 0], [1, 0, 0], [0, 0, -1]]) @ R_p @ np.array([[0, 1, 0], [1, 0, 0], [0, 0, -1]])
print(f"R_agi post-transform:\n{R_rc_agi}")
t_rc_agi = np.array([[0, 1, 0], [1, 0, 0], [0, 0, -1]]) @ ref_df.iloc[i, 10:13].to_list()
fl = cam.calibration.f
cx, cy = cam.calibration.cx + (1648//2), cam.calibration.cy + (1200//2)
b1, b2 = cam.calibration.b1, cam.calibration.b2

hva_cam = np.array([[fl+b1, 0, 0], [b2, fl, 0], [cx, cy, 1]])

# transform from agisoft camera frame to rover frame
c_r_est = t_rc_agi
hva_r_est = R_rc_agi @ hva_cam
h_r_est, v_r_est, a_r_est  = hva_r_est[:,0], hva_r_est[:,1], hva_r_est[:,2]
o_r_est = a_r_est.copy()
k1, k2 = cam.calibration.k1, cam.calibration.k2
r_r_est = np.array([0, k1, k2])

Hr_est = np.array([h_r_est, v_r_est, a_r_est]).T
# print(f"Hr_est:\n{Hr_est}")


#======================   PDS Label   =====================
# parse cahvor from PDS label
IMG_path = glob.glob( directory_input + '/zcam/*/' + cam.label[:54] + '.IMG' ) +\
            glob.glob( directory_input + '/zcam/' + cam.label[:54] + '.IMG' )
im = Image(IMG_path=IMG_path[0])
c_r = np.array(im.label['GEOMETRIC_CAMERA_MODEL']['MODEL_COMPONENT_1'])
a_r = np.array(im.label['GEOMETRIC_CAMERA_MODEL']['MODEL_COMPONENT_2'])
h_r = np.array(im.label['GEOMETRIC_CAMERA_MODEL']['MODEL_COMPONENT_3'])
v_r = np.array(im.label['GEOMETRIC_CAMERA_MODEL']['MODEL_COMPONENT_4'])
o_r = np.array(im.label['GEOMETRIC_CAMERA_MODEL']['MODEL_COMPONENT_5'])
r_r = np.array(im.label['GEOMETRIC_CAMERA_MODEL']['MODEL_COMPONENT_6'])

# Extrinsic and intrinsic camera parameters from PDS labels
x, y, z = c_r[0], c_r[1], c_r[2]

hs = np.linalg.norm(np.cross(a_r, h_r), ord=2)
vs = np.linalg.norm(np.cross(a_r, v_r), ord=2)
hc = np.dot(h_r, a_r)
vc = np.dot(v_r, a_r)

hp = (h_r-hc*a_r)/hs
vp = (v_r-vc*a_r)/vs
norm_hp = np.linalg.norm(hp, ord=2)
norm_vp = np.linalg.norm(vp, ord=2)

sin_theta = np.clip(np.linalg.norm( np.cross(vp, hp)), a_min=-1, a_max=1)
cos_theta = np.sqrt(1-sin_theta**2)

fl = vs
b1 = - (hs * (-sin_theta)) - vs
b2 = hs * cos_theta
cx = hc
cy = vc
k0, k1, k2 = r_r[0], r_r[1], r_r[2]

Hc = np.array([[fl+b1, 0, 0],
            [b2, fl, 0],
            [cx, cy, 1]])
Hr = np.array([h_r, v_r, a_r]).T
R_rc_pds = Hr @ np.linalg.inv(Hc)

print(f"R_pds:\n{R_rc_pds}")

# print(f"Hr:\n{Hr}")

ZR0_1178_0771519008_270RAD_N0540000ZCAM09220_0630LMA01
R_agi pre-transform:
[[-0.55814379  0.77637334 -0.2927797 ]
 [-0.6745053  -0.63003551 -0.38483483]
 [-0.48323711 -0.01731171  0.87531834]]
R_agi post-transform:
[[-0.63003551 -0.6745053   0.38483483]
 [ 0.77637334 -0.55814379  0.2927797 ]
 [ 0.01731171  0.48323711  0.87531834]]
R_pds:
[[-0.62673139 -0.66618312  0.398264  ]
 [ 0.77917397 -0.55095373  0.307003  ]
 [ 0.01202499  0.50262703  0.864379  ]]
